# PerovSeek

配方推荐、光谱预测与器件验证反馈。

Formulation recommendation, spectral prediction and device feedback.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from perovseek.bayesian import recommend
from perovseek.spectra import (
    load_spectra, SpectralPredictor, plot_spectra,
    plot_predictions, prediction_metrics,
)

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
OUTPUT = PROJECT / "outputs"
OUTPUT.mkdir(exist_ok=True)


## 1. 贝叶斯优化

Bayesian optimization

读取九组分配方、实测 PCE 与搜索边界。

Read nine-component formulations, measured PCE and search bounds.


In [ ]:
DATA_FILE = PROJECT / "data/formulations.xlsx"
formulations = pd.read_excel(DATA_FILE, sheet_name="Formulations")
bounds = pd.read_excel(DATA_FILE, sheet_name="Bounds")
TARGET = "PCE"

print("Measured formulations:", len(formulations))
display(formulations.head(4).round(3))


In [ ]:
parameters = {
    "batch_size": 6,
    "seed": 42,
    "noise_sd": 2.0,
    "mc_samples": 128,
    "num_restarts": 4,
    "raw_samples": 128,
    "fit_maxiter": 40,
    "opt_maxiter": 80,
}


In [ ]:
candidates = recommend(
    formulations, target=TARGET, bounds=bounds, **parameters
)
candidates.to_csv(OUTPUT / "recommended_formulations.csv", index=False)
candidate_view = candidates.set_index("candidate_id").drop(
    columns="PCE_std"
)
display(candidate_view.round(3))


## 2. 高通量实验与表征

High-throughput experimentation and characterization

制备候选薄膜，采集吸收与 PL 光谱，记录样本编号。

Prepare the films, acquire absorption and PL spectra, and record sample identifiers.


## 3. 预训练模型预测

Pretrained model prediction

读取已有样本的吸收与双面激发 PL 光谱。

Read recorded absorption and PL spectra under top and bottom excitation.


In [ ]:
SPECTRA_FILE = PROJECT / "data/spectra/1.59eV_additive_data.xlsx"
spectra = load_spectra(SPECTRA_FILE)
print("Spectral samples:", len(spectra))
spectrum_figure = plot_spectra(spectra)
spectrum_figure.savefig(OUTPUT / "measured_spectra.png", dpi=180)
display(spectrum_figure)


In [ ]:
CHECKPOINT = PROJECT / "checkpoints/spectral_pce_state.pt"
predictor = SpectralPredictor(CHECKPOINT)
predictions = predictor.predict(spectra, subset="test")
predictions.to_csv(OUTPUT / "spectral_predictions.csv", index=False)

display(predictions.head(5).round(3))


In [ ]:
metrics = prediction_metrics(predictions)
prediction_figure = plot_predictions(predictions)
prediction_figure.savefig(OUTPUT / "spectral_pce_scatter.png", dpi=180)
display(prediction_figure)
display(pd.Series(metrics, name="Test results"))


In [ ]:
ranked = predictions.sort_values("predicted_pce_percent", ascending=False).head(8)
ranked.to_csv(OUTPUT / "ranked_spectral_samples.csv", index=False)
display(ranked.round(2))


## 4. 器件验证与反馈

Device validation and feedback

新配方的器件实测 PCE 用于下一轮优化，待测值保持空白。

Measured device PCE feeds the next optimization round; pending values remain empty.


In [ ]:
feedback = candidates.drop(columns=["PCE_pred", "PCE_std"]).copy()
feedback["PCE"] = np.nan
feedback["measurement_status"] = "not measured"
feedback.to_csv(OUTPUT / "device_feedback.csv", index=False)

display(feedback[["candidate_id", "PCE", "measurement_status"]].head(3))
